# 01. Significance of Classification Performance (Appendix E.1; Table A1; Appendix D.2.2, Tables D1–D2)

> Reproduction notebook for *Addressing the Psychological Small-Sample Problem with Multimodal Large Language Models: Evidence from Multiple Moral Emotion Recognition Study*. Section numbers refer to the manuscript appendices. Every notebook reads only the committed files in `../final/` and ends with a check against the frozen outputs in `../final/reproducibility/`. Run the notebooks in numerical order; `00` writes the label tables that the others reuse.

Implements Appendix E.1 for every configuration with participant-level predictions (Exp 1–13 and the supplementary control Exp 16) and for the two observer passes (Exp 14–15), and recomputes the per-model statistics of the still-frame comparison (Exp 17) reported in Appendix D.2.2.

* **E.1.1 Scoring and confidence intervals.** Four-class accuracy (chance .25), macro-F1 and Cohen's κ, with participant-level bootstrap intervals (10,000 resamples; κ on 2,000) taken at the 2.5th and 97.5th percentiles. Two-dimensional accuracy (mean of the sympathy-axis and inspiration-axis accuracies, chance .50) is reported alongside.
* **E.1.2 Significance tests.** One-sided permutation test: the design labels are shuffled against the fixed predictions 10,000 times and *p* = (1 + #{permuted accuracy ≥ observed}) / (10,000 + 1). False-discovery-rate correction (Benjamini–Hochberg) is applied across the 14 model configurations (Exp 1–13 and 16), as in Table A1 and Appendix E.1.2.5; the observer passes (Exp 14–15) are tested by the same permutation procedure but sit outside that family and receive no FDR-adjusted *p*. Note that the `p_fdr` column of the frozen file `significance_frozen.csv` was computed by applying Benjamini–Hochberg to all 16 permutation *p* values (observer passes included); this reproduces its 14 model-configuration rows (and the Exp 15 row), but not its Exp 14 row (.150 vs .131), whose origin is not recoverable; the verification therefore checks the FDR values against the 14-member family applied to the frozen permutation *p* values, which are the values printed in Table A1. For the few-shot (Exp 7), fine-tuned (Exp 8) and Random Forest (Exp 10–13) configurations the test does not re-run retrieval or training under each permuted labelling.
* **E.1.3 Confusion structures and error patterns.** Per-class recall (Appendix Figure E1) and the confusion matrices behind the counts quoted in E.1.3.1–E.1.3.5.
* **E.1.4 Modality-specific results for conventional machine learning.** Per-modality accuracy, axis accuracy and per-class recall of the single-modality Random Forest predictions (Appendix Figure E2).
* **Appendix D.2.2, Tables D1–D2.** Per-model statistics for the still-frame comparison (Exp 17), recomputed from the saved confusion matrices by expanding each matrix back into label vectors.

In [1]:
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("."))
import common as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(C.SEED)
print("final data folder:", os.path.relpath(C.FINAL))

final data folder: ../final


## 1. Load the label tables

In [2]:
ALL = pd.read_csv(os.path.join(C.OUT, "labels_all.csv"))
LAB = {e: g.drop(columns="exp").reset_index(drop=True) for e, g in ALL.groupby("exp")}
print({e: len(LAB[e]) for e in sorted(LAB)})

{1: 96, 2: 100, 3: 100, 4: 100, 5: 100, 6: 100, 7: 100, 8: 100, 9: 99, 10: 101, 11: 103, 12: 103, 13: 103, 14: 100, 15: 100, 16: 100}


## 2. E.1.1–E.1.2: bootstrap intervals, permutation tests, FDR (Table A1)

In [3]:
from statsmodels.stats.multitest import multipletests
rows = []
for e in sorted(LAB):
    r = C.significance_battery(LAB[e], rng); r["exp"] = e; r["name"] = C.NAMES[e]; rows.append(r)
SIG = pd.DataFrame(rows).set_index("exp")
fam = [e for e in SIG.index if e not in C.OBS_EXPS]
print("FDR family:", fam, "| m =", len(fam))
SIG["p_fdr"] = np.nan; SIG.loc[fam, "p_fdr"] = multipletests(SIG.loc[fam, "p_perm"], method="fdr_bh")[1]
SIG["p_fdr_dim"] = np.nan; SIG.loc[fam, "p_fdr_dim"] = multipletests(SIG.loc[fam, "p_perm_dim"], method="fdr_bh")[1]
SIG.round(4).to_csv(os.path.join(C.OUT, "significance.csv"))
SIG[["name", "n", "acc", "acc_lo", "acc_hi", "macro_f1", "kappa", "kappa_lo", "kappa_hi", "p_perm", "p_fdr", "dim_acc", "symp_axis", "insp_axis", "p_perm_dim"]].round(3)

FDR family: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 16] | m = 14
Out[3]: 
                       name    n    acc  acc_lo  acc_hi  macro_f1  kappa  kappa_lo  kappa_hi  p_perm  p_fdr  dim_acc  symp_axis  insp_axis  p_perm_dim
exp                                                                                                                                                   
1               Baseline(V)   96  0.333   0.240   0.427     0.242  0.091     0.005     0.187   0.020  0.021    0.578      0.604      0.552       0.003
2                       V+A  100  0.490   0.390   0.590     0.444  0.311     0.188     0.435   0.000  0.000    0.680      0.710      0.650       0.000
3                     V+A+T  100  0.630   0.530   0.720     0.606  0.502     0.380     0.624   0.000  0.000    0.785      0.810      0.760       0.000
4                   V+A+T+P  100  0.690   0.600   0.780     0.689  0.587     0.460     0.704   0.000  0.000    0.835      0.860      0.810       0.000
5               

## 3. E.1.3: per-class recall and confusion matrices

Rows of each confusion matrix are the induced (true) classes, columns the predicted classes. The matrices printed here are the source of the counts quoted in Appendix E.1.3.1–E.1.3.5 (Exp 1, 8 and 9) and E.4.6 (Exp 15).

In [4]:
REC = pd.DataFrame({e: C.per_class_recall(LAB[e]) for e in sorted(LAB)}).T
REC["weakest"] = REC[C.CLS].idxmin(1); REC["strongest"] = REC[C.CLS].idxmax(1)
REC.round(4).to_csv(os.path.join(C.OUT, "perclass_recall.csv"))
CM = {e: C.confusion(LAB[e]) for e in (1, 2, 4, 8, 9, 15)}
for e, cm in CM.items(): print(f"Exp {e} ({C.NAMES[e]}) confusion matrix (rows = true, columns = predicted):"); print(cm); print()
REC.round(2)

Exp 1 (Baseline(V)) confusion matrix (rows = true, columns = predicted):
col_0  ELVT  SYMP  INSP  CTRL
row_0                        
ELVT      0     6     0    16
SYMP      0     6     0    18
INSP      0     1     3    20
CTRL      0     3     0    23

Exp 2 (V+A) confusion matrix (rows = true, columns = predicted):
col_0  ELVT  SYMP  INSP  CTRL
row_0                        
ELVT      2     6     7     8
SYMP      0    15     4     6
INSP      1     1    12    10
CTRL      0     2     6    20

Exp 4 (V+A+T+P) confusion matrix (rows = true, columns = predicted):
col_0  ELVT  SYMP  INSP  CTRL
row_0                        
ELVT     17     6     0     0
SYMP      4    17     1     3
INSP      7     0    15     2
CTRL      1     2     5    20

Exp 8 (FineTune) confusion matrix (rows = true, columns = predicted):
col_0  ELVT  SYMP  INSP  CTRL
row_0                        
ELVT     12     3     4     4
SYMP      1    18     1     5
INSP      5     0    12     7
CTRL      1     2     1    24


## 4. E.1.4: single-modality Random Forest predictions (Appendix Figure E2)

One vote per participant and modality (acoustics: the run with correlation pruning, Appendix C.2.4).

In [5]:
ml = C.load_ml_predictions(); rowsA = []; rowsB = []
for mod, g in ml.groupby("modality"):
    sa, ia, _ = C.axis_accuracy(g.true, g.pred)
    rowsA.append(dict(modality=mod, N=len(g), acc=(g.true == g.pred).mean(), symp_axis=sa, insp_axis=ia))
    rec = C.per_class_recall(g); rec["modality"] = mod; rowsB.append(rec)
MA = pd.DataFrame(rowsA).sort_values("acc", ascending=False); MB = pd.DataFrame(rowsB).set_index("modality")[C.CLS]
MA.to_csv(os.path.join(C.OUT, "ml_modality_axis.csv"), index=False); MB.to_csv(os.path.join(C.OUT, "ml_modality_recall.csv"))
print(MA.round(3).to_string(index=False)); MB.round(2)

    modality   N   acc  symp_axis  insp_axis
     thermal  90 0.422      0.756      0.589
      Facial  99 0.394      0.697      0.535
       r-PPG  93 0.376      0.591      0.495
 Speech Text 101 0.347      0.475      0.584
        pose  99 0.313      0.515      0.556
  Accoustics 101 0.257      0.505      0.515
Eye_Movement 101 0.198      0.525      0.465
Out[5]: 
              ELVT  SYMP  INSP  CTRL
modality                            
Accoustics    0.22  0.28  0.29  0.24
Eye_Movement  0.21  0.21  0.12  0.24
Facial        0.17  0.39  0.42  0.57
Speech Text   0.35  0.20  0.25  0.55
pose          0.29  0.13  0.33  0.46
r-PPG         0.29  0.48  0.58  0.14
thermal       0.17  0.57  0.36  0.58


## 5. Appendix D.2.2 (Tables D1–D2): still-frame comparison (Exp 17), six models and their joint average

The notebook `experiment_17_18.ipynb` ships with outputs cleared; its confusion matrices are stored in `exp17_18_confusion_matrices.json`. The key `confusion_matrices_exp18` holds manuscript Exp 17 (the notebook's run labelled *Experiment 18*, stimulus-viewing recordings); the key `confusion_matrices_exp17` holds the unreported post-viewing run and is not analysed here. Accuracy, macro-F1, κ and the permutation *p* are recomputed from the matrices.

In [6]:
from sklearn.metrics import f1_score, cohen_kappa_score
J = json.load(open(os.path.join(C.FINAL, "exp17_18_confusion_matrices.json"), encoding="utf-8"))
rows = []
for model, cm in J["confusion_matrices_exp18"].items():
    t, p = C.expand_confusion(cm); acc = (t == p).mean()
    perm = np.array([(rng.permutation(t) == p).mean() for _ in range(C.NP)])
    rows.append(dict(model=model, n=len(t), accuracy=acc, macro_f1=f1_score(t, p, labels=C.CLS, average="macro"),
                     kappa=cohen_kappa_score(t, p, labels=C.CLS), p_perm=(np.sum(perm >= acc) + 1) / (C.NP + 1)))
EXP17 = pd.DataFrame(rows); EXP17.round(3).to_csv(os.path.join(C.OUT, "exp17_per_model.csv"), index=False); EXP17.round(3)

Out[6]: 
                model    n  accuracy  macro_f1  kappa  p_perm
0    Claude Haiku 4.5  100      0.29     0.236  0.026   0.334
1  Deepseek 4.1 Flash  100      0.25     0.194 -0.024   0.729
2           GPT6 Luna  100      0.30     0.210  0.036   0.223
3    Gemini 2.5 Flash  100      0.30     0.257  0.047   0.217
4    Gemini 3.8 Flash  100      0.30     0.244  0.051   0.196
5      Qwen 3.8 Flash  100      0.28     0.227  0.020   0.399
6       Joint average  100      0.31     0.295  0.067   0.135


## 6. Verification

Table A1, per-class recall and the single-modality statistics are compared with the frozen outputs. The FDR column is not compared with the frozen `p_fdr` column, which was computed by applying Benjamini–Hochberg to all 16 permutation *p* values (observer passes included); this reproduces its 14 model-configuration rows (and the Exp 15 row), but not its Exp 14 row (.150 vs .131), whose origin is not recoverable; instead the Benjamini–Hochberg procedure is applied to the frozen permutation *p* values of the 14-member family, which reproduces Table A1 (for example Exp 1 .024, Exp 11 .014, Exp 12 .020), and the recomputed FDR values are compared with that reference. The confusion-matrix counts of Appendix E.1.3 and the Table D1 entries have no frozen file; they are compared with the values printed in the manuscript (Table D1 reports accuracy, macro-F1 and κ to two decimals and the permutation *p* to two decimals; Table D2 repeats the accuracy and macro-F1 columns alongside the axis-level statistics).

In [7]:
fz = C.load_frozen("significance_frozen.csv").rename(columns={"Unnamed: 0": "exp"})
print(C.compare(SIG.reset_index(), fz, ["exp"], ["n", "acc", "acc_lo", "acc_hi", "macro_f1", "kappa", "kappa_lo", "kappa_hi", "p_perm", "dim_acc", "symp_axis", "insp_axis"], atol=0.011).to_string(index=False))
# FDR: the frozen p_fdr column was computed with Benjamini-Hochberg over all 16 permutation p values (it reproduces the family rows, not the Exp 14 row); Table A1 uses the 14-member family (Exp 1-13, 16)
fzi = fz.set_index("exp"); fam14 = [e for e in fzi.index if e not in C.OBS_EXPS]
bh16 = pd.Series(multipletests(fzi.p_perm, method="fdr_bh")[1], index=fzi.index)
print("frozen p_fdr, 14 family rows, equals BH computed over all", len(fzi), "permutation p values:", bool(np.allclose(fzi.loc[fam14, "p_fdr"], bh16.loc[fam14], atol=1e-4)),
      "| observer rows (frozen vs BH-16):", {e: (round(float(fzi.loc[e, "p_fdr"]), 4), round(float(bh16.loc[e]), 4)) for e in C.OBS_EXPS})
REF = pd.Series(multipletests(fzi.loc[fam14, "p_perm"], method="fdr_bh")[1], index=fam14, name="p_fdr_ref")
print("Table A1 p (FDR), 14-member family on the frozen permutation p (Exp 1, 11, 12):", REF.loc[[1, 11, 12]].round(3).tolist(), "(Table A1: .024, .014, .020)")
d_fdr = (SIG.loc[fam14, "p_fdr"] - REF).abs()
print(f"recomputed p_fdr vs 14-family reference: {len(d_fdr)} configurations, max |diff| {d_fdr.max():.4f}, within tolerance (.011): {bool(d_fdr.max() <= 0.011)}; observer rows without p_fdr: {SIG.loc[C.OBS_EXPS, 'p_fdr'].isna().all()}")
fr = C.load_frozen("perclass_recall_frozen.csv").rename(columns={"Unnamed: 0": "exp"})
print(C.compare(REC.reset_index().rename(columns={"index": "exp"}), fr, ["exp"], C.CLS, atol=0.001).to_string(index=False))
print(C.compare(MA, C.load_frozen("ml_modality_axis_5fold.csv"), ["modality"], ["N", "acc", "symp_axis", "insp_axis"], atol=0.001).to_string(index=False))
print(C.compare(MB.reset_index(), C.load_frozen("ml_modality_recall_5fold.csv"), ["modality"], C.CLS, atol=0.001).to_string(index=False))
# Appendix E.1.3: counts quoted in the text (true class -> predicted class)
E13 = [(1, "ELVT", "CTRL", 16), (1, "SYMP", "CTRL", 18), (1, "INSP", "CTRL", 20), (8, "CTRL", "CTRL", 24), (8, "INSP", "ELVT", 5), (8, "ELVT", "INSP", 4), (8, "INSP", "CTRL", 7),
       (9, "CTRL", "CTRL", 24), (9, "CTRL", "ELVT", 0), (9, "CTRL", "SYMP", 0), (9, "ELVT", "SYMP", 5), (9, "SYMP", "SYMP", 20), (9, "INSP", "INSP", 17), (15, "ELVT", "ELVT", 8), (15, "CTRL", "CTRL", 27)]
chk = [(e, t, p, v, int(CM[e].loc[t, p])) for e, t, p, v in E13]
print("E.1.3 / E.4.6 confusion counts matching the manuscript:", sum(v == c for *_, v, c in chk), "/", len(chk), [x for x in chk if x[3] != x[4]])
print("E.1.3.4 Exp 8 recall INSP / ELVT:", round(REC.loc[8, "INSP"], 2), "/", round(REC.loc[8, "ELVT"], 2), "(manuscript .50 / .52)")
A4 = {"Claude Haiku 4.5": (.29, .24, .03, .33), "Deepseek 4.1 Flash": (.25, .19, -.02, .73), "GPT6 Luna": (.30, .21, .04, .23), "Gemini 2.5 Flash": (.30, .26, .05, .21),
      "Gemini 3.8 Flash": (.30, .24, .05, .20), "Qwen 3.8 Flash": (.28, .23, .02, .39), "Joint average": (.31, .30, .07, .13)}
ms = pd.DataFrame(A4, index=["accuracy", "macro_f1", "kappa", "p_perm"]).T.rename_axis("model").reset_index()
print(C.compare(EXP17.round(2), ms, ["model"], ["accuracy", "macro_f1", "kappa", "p_perm"], atol=0.011).to_string(index=False))

   column  n_compared  max_abs_diff  within_tolerance
        n          16      0.000000              True
      acc          16      0.000042              True
   acc_lo          16      0.010000              True
   acc_hi          16      0.010000              True
 macro_f1          16      0.000050              True
    kappa          16      0.000049              True
 kappa_lo          16      0.010042              True
 kappa_hi          16      0.008117              True
   p_perm          16      0.002202              True
  dim_acc          16      0.000045              True
symp_axis          16      0.000045              True
insp_axis          16      0.000045              True
frozen p_fdr, 14 family rows, equals BH computed over all 16 permutation p values: True | observer rows (frozen vs BH-16): {14: (0.1495, 0.1308), 15: (0.0002, 0.0002)}
Table A1 p (FDR), 14-member family on the frozen permutation p (Exp 1, 11, 12): [0.024, 0.014, 0.02] (Table A1: .024, .014, .020)
